# Train a text intent model from scratch for Arduino UNO Q
This is the text-request component, separate from the visual classifier. Randomly initialized NumPy softmax model; no pretrained weights or AI service. CPU training is sufficient even inside your Ampere GPU VM. Inference on the board uses standard Python only.

Initial scope: search links, arithmetic, clarification and rejection. Search prepares a URL rather than autonomously browsing. This is not a generative language model or universal complexity estimator.


In [ ]:
from pathlib import Path
import sys, subprocess, json
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "local_model").is_dir() and (PROJECT_ROOT.parent / "local_model").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "local_model/train_intent.py").is_file(), "Set PROJECT_ROOT to the updated project directory."
def run_module(module, *args):
    subprocess.run([sys.executable, "-u", "-m", module, *map(str,args)], cwd=PROJECT_ROOT, check=True)
print(PROJECT_ROOT)


## Define the training data
The bundled JSONL contains synthetic text/label pairs. Edit or append your examples using search, compute or reject labels. This cell only generates data if missing, preserving edits. Keep independently collected test prompts separate from training. Supporting a new action also requires an executor and evaluation; adding a label alone does not implement it.


In [ ]:
DATA = PROJECT_ROOT / "data/intent_training.jsonl"
if not DATA.exists():
    run_module("local_model.intent_dataset", "--out", DATA)
rows = [json.loads(line) for line in DATA.read_text().splitlines() if line.strip()]
print("Training examples:", len(rows))
print(rows[:3])
# To add a labeled example deliberately, edit DATA in Jupyter's text editor.


In [ ]:
OUTPUT = PROJECT_ROOT / "artifacts_intent"
EPOCHS = 400
SEED = 7
run_module("local_model.train_intent", "--data", DATA, "--outdir", OUTPUT,
           "--epochs", EPOCHS, "--seed", SEED)
report = json.loads((OUTPUT / "intent_training_report.json").read_text())
print(json.dumps(report, indent=2))


## Check requests without a UI
The trained classifier selects an intent; bounded code extracts arguments and calculates results. Unrecognized phrasing and uncertain classifications are refused. These examples do not establish broad language understanding.


In [ ]:
for request in ["Compute 2 + 2", "Search Nvidia", "Generate an image of the Empire State Building"]:
    print("Request:", request)
    run_module("local_model.text_assistant", "--model", OUTPUT / "intent_model.json", request)


## Package for offline UNO Q inference
Download uno-q-text.zip and transfer it over USB using ADB. No cross compilation, GPU, pip install, or network is required on the board. Extract and run python3 run_assistant.py inside uno-q-text. Model weights persist; live requests are not saved by the application. Terminal/OS copies are outside secure-erasure guarantees.


In [ ]:
subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts/package_text.py"),
                "--model", str(OUTPUT / "intent_model.json"),
                "--out", str(PROJECT_ROOT / "uno-q-text.zip")], cwd=PROJECT_ROOT, check=True)
